# IssueCommentEvent pipeline test (`04_issue_comment_events`)

This notebook tests `04_issue_comment_events` with a small synthetic Bronze table in the schema `github_lakehouse.test_issue_comment`. It runs the real notebook with `%run` and `test_schema = test_issue_comment`, so **production tables are never written**. The last check section proves that by comparing the production table versions before and after.

**Before you run it**, the production tables must exist, because the test tables are created with `CREATE TABLE ... LIKE`:

1. `11_silver_audit_table` (audit table; it already exists from the PushEvent work)
2. cells 1–6 of `04_issue_comment_events` (they create `silver.issue_comment_events` and `quarantine.quarantine_issue_comment_event`)

Every check prints `PASS` or `FAIL`. A section stops with an error if any of its checks failed. The last cell drops the test schema.

The shared pipeline (parse errors, core IDs, duplicates, all-dates mode, partial failure) is already covered by `tests/test_08_fork_events`. This test covers what is specific to IssueCommentEvent: the three source shapes the manual asks for (a normal issue comment, a pull request conversation comment, and an unclassifiable target), the other `comment_target_type` edge cases, missing or invalid IDs kept with warnings, an unexpected action kept with an `UNEXPECTED_VALUE` warning, two events about the same `comment_id` both kept, and the quarantine table keeping the comment fields and their original text.

## B1. Setup: test schema, empty tables and helpers

In [0]:
import json
from datetime import date, datetime, timezone

TEST_SCHEMA = "test_issue_comment"
assert TEST_SCHEMA.startswith("test_")   # never drop anything outside a test schema

def T(name):
    return f"github_lakehouse.{TEST_SCHEMA}.{name}"

TEST_BRONZE = T("github_events_raw")
TEST_SILVER = T("issue_comment_events")
TEST_QUARANTINE = T("quarantine_issue_comment_event")
TEST_AUDIT = T("silver_pipeline_runs")

PROD_TABLES = [
    "github_lakehouse.bronze.github_events_raw",
    "github_lakehouse.silver.issue_comment_events",
    "github_lakehouse.quarantine.quarantine_issue_comment_event",
    "github_lakehouse.audit.silver_pipeline_runs",
]

# The test tables copy the production definitions, so those must exist first.
missing = [t for t in PROD_TABLES if not spark.catalog.tableExists(t)]
if missing:
    raise RuntimeError(
        f"Missing production tables: {missing}. Run 11_silver_audit_table "
        "and cells 1-6 of 04_issue_comment_events (table creation) first."
    )

spark.sql(f"CREATE SCHEMA IF NOT EXISTS github_lakehouse.{TEST_SCHEMA}")
for t in [TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

# Bronze: only the four columns the pipeline reads.
spark.sql(f"""
    CREATE TABLE {TEST_BRONZE} (
        raw_json STRING, _source_file STRING,
        _source_date DATE, _ingested_at TIMESTAMP
    ) USING DELTA
""")
# Targets: exact copies of the production table definitions (no data).
spark.sql(f"CREATE TABLE {TEST_SILVER} LIKE github_lakehouse.silver.issue_comment_events")
spark.sql(f"CREATE TABLE {TEST_QUARANTINE} LIKE github_lakehouse.quarantine.quarantine_issue_comment_event")
spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE github_lakehouse.audit.silver_pipeline_runs")


# ---- assertion helpers ----
_results = []

def check(label, actual, expected):
    ok = actual == expected
    _results.append((label, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {label}: actual={actual!r} expected={expected!r}")

def finish(section):
    failed = [label for label, ok in _results if not ok]
    _results.clear()
    if failed:
        raise AssertionError(f"{section}: {len(failed)} check(s) failed: {failed}")
    print(f"\n{section}: ALL CHECKS PASSED")

def check_audit(expected_counts, expected_quality=None,
                expected_status="SUCCEEDED", reconciled=True):
    # `audit` is the dict created by the pipeline's audit cell via %run.
    check("audit status", audit["status"], expected_status)
    check("audit reconciliation_passed", audit["reconciliation_passed"], reconciled)
    for k, v in expected_counts.items():
        check(f"audit {k}", audit.get(k), v)
    for k, v in (expected_quality or {}).items():
        check(f"quality {k}", audit["quality_metrics"].get(k), v)

def rows(table, where="true"):
    return [r.asDict() for r in spark.table(table).where(where).collect()]

def one_silver(event_id):
    found = rows(TEST_SILVER, f"event_id = '{event_id}'")
    check(f"silver has exactly one '{event_id}'", len(found), 1)
    return found[0] if found else {}

def table_version(t):
    return spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").collect()[0]["version"]

print("Setup complete:", TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT)

## B2. Safety snapshot of the production tables

In [0]:
prod_versions_before = {t: table_version(t) for t in PROD_TABLES}
prod_versions_before

## B3. Load the synthetic Bronze data

The default comment is on issue `id 8001`, `number 21`, with `comment.id 9001` and action `created`, and has no `pull_request` URL.

| Row | Case | Expected result |
|---|---|---|
| `c1` | normal issue comment, `comment.id 9111` | Silver, target `issue`, no warning |
| `c2` | pull request conversation comment (`issue.pull_request.url` set) | Silver, target `pull_request`, no warning |
| `c3` | no `issue` object at all | Silver, target NULL, warnings `MISSING: issue_id; MISSING: issue_number; UNCLASSIFIED: comment_target_type` |
| `c4` | issue has a number but no id | Silver, target `issue`, warning `MISSING: issue_id` |
| `c5` | `pull_request.url` is blank | Silver, target `issue` (blank URL = no URL), no warning |
| `c6` | URL set, but no issue id or number | Silver, target `pull_request`, warnings `MISSING: issue_id; MISSING: issue_number` |
| `c7` | `comment.id` missing | Silver, warning `MISSING: comment_id` |
| `c8` | `action` missing | Silver, warning `MISSING: comment_action` |
| `c9` | `action = edited` | Silver, value kept, warning `UNEXPECTED_VALUE: comment_action` |
| `c10` | `payload = {}` (like the 2 profiled rows) | Silver, five warnings, target NULL |
| `c11` | a second event about comment `9111` | Silver; both `c1` and `c11` are kept (dedup is by `event_id` only) |
| `c12` | `comment` is text, not an object | quarantine `JSON_PARSE_ERROR` |
| `c13` | `repo.id = "x"` | quarantine `INVALID_CAST: repo_id`, comment fields and original text kept |
| `c14` (twice) | duplicate `event_id`, older and newer ingestion | one Silver row, from the newer file |
| `c15` | `issue.id = "abc"`, number present | Silver, target `issue`, warning `INVALID_CAST: issue_id` |
| `x1` | an IssuesEvent | ignored (not read by this notebook) |

In [0]:
F1 = "synthetic/2024-02-01-0.json.gz"
F2 = "synthetic/2024-02-01-1.json.gz"
D1 = date(2024, 2, 1)
I1 = datetime(2024, 3, 1, tzinfo=timezone.utc)   # earlier ingestion
I2 = datetime(2024, 3, 2, tzinfo=timezone.utc)   # later ingestion
PR_URL = "https://api.github.com/repos/octo/repo/pulls/22"

def issue(drop=(), **changes):
    """The issue wrapper of the comment. Defaults = a plain issue (no PR URL)."""
    i = {"id": 8001, "number": 21}
    i.update(changes)
    for key in drop:
        i.pop(key)
    return i

def ev(event_id, type_="IssueCommentEvent", created_at="2024-02-01T10:00:00Z",
       actor=None, repo=None, action="created", iss=None, comment=None, payload=None):
    """One GitHub-style event as a JSON string. Defaults = a clean issue comment."""
    event = {
        "id": event_id,
        "type": type_,
        "created_at": created_at,
        "actor": {"id": 101, "login": "alice"} if actor is None else actor,
        "repo": {"id": 201, "name": "octo/repo"} if repo is None else repo,
        "payload": ({"action": action,
                     "issue": issue() if iss is None else iss,
                     "comment": {"id": 9001} if comment is None else comment}
                    if payload is None else payload),
    }
    return json.dumps(event)

synthetic_rows = [
    # (raw_json, _source_file, _source_date, _ingested_at)                     # case
    (ev("c1", comment={"id": 9111}),                            F1, D1, I1),   # issue comment
    (ev("c2", iss=issue(pull_request={"url": PR_URL})),          F1, D1, I1),   # PR conversation comment
    (ev("c3", payload={"action": "created",
                       "comment": {"id": 9003}}),                F1, D1, I1),   # unclassifiable target
    (ev("c4", iss=issue(drop=["id"])),                           F1, D1, I1),   # number only -> issue
    (ev("c5", iss=issue(pull_request={"url": ""})),              F1, D1, I1),   # blank URL -> issue
    (ev("c6", iss={"pull_request": {"url": PR_URL}}),            F1, D1, I1),   # URL, no issue ids
    (ev("c7", comment={}),                                       F1, D1, I1),   # comment_id missing
    (ev("c8", payload={"issue": issue(),
                       "comment": {"id": 9008}}),                F1, D1, I1),   # action missing
    (ev("c9", action="edited"),                                  F1, D1, I1),   # unexpected action
    (ev("c10", payload={}),                                      F1, D1, I1),   # empty payload
    (ev("c11", comment={"id": 9111}),                           F1, D1, I1),   # same comment_id as c1
    (ev("c12", comment="oops"),                                  F1, D1, I1),   # parse error
    (ev("c13", repo={"id": "x", "name": "octo/repo"}),           F1, D1, I1),   # repo_id invalid
    (ev("c14", comment={"id": 1401}),                            F1, D1, I1),   # duplicate, older
    (ev("c14", comment={"id": 1402}),                            F2, D1, I2),   # duplicate, newer -> wins
    (ev("c15", iss=issue(id="abc")),                             F1, D1, I1),   # issue_id invalid
    (ev("x1", type_="IssuesEvent"),                              F1, D1, I1),   # other event type
]

(spark.createDataFrame(synthetic_rows, spark.table(TEST_BRONZE).schema)
    .write.mode("overwrite").saveAsTable(TEST_BRONZE))

check("synthetic bronze rows", spark.table(TEST_BRONZE).count(), 17)
finish("B3 load")

## B4. Run 1: single date 2024-02-01

In [0]:
%run ../04_issue_comment_events $test_schema="test_issue_comment" $source_date="2024-02-01" $run_id="test-run1"

## B5. Run 1 checks

In [0]:
QUALITY_RUN1 = {
    "null_event_timestamp": 0, "null_actor_id": 0, "null_actor_login": 0,
    "null_repo_name": 0,
    "null_comment_action": 2, "null_issue_id": 5, "null_issue_number": 3,
    "null_issue_id_and_number": 3, "null_comment_target_type": 2,
    "null_comment_id": 2, "unexpected_comment_action": 1,
    "missing_lineage": 0,
}

# --- tables and audit identity: the run used the test tables only
check("resolved silver", SILVER_TABLE, TEST_SILVER)
check("resolved quarantine", QUARANTINE_TABLE, TEST_QUARANTINE)
check("audit pipeline_name", audit["pipeline_name"], "silver_issue_comment_events")
check("audit event_type", audit["event_type"], "IssueCommentEvent")

# --- audit: 16 IssueCommentEvent rows (x1 is an IssuesEvent)
check_audit({
    "bronze_input_rows": 16, "hard_null_quarantine_rows": 2,
    "validation_quarantine_rows": 0, "validated_rows": 14,
    "duplicate_rows_removed": 1, "silver_candidate_rows": 13,
    "silver_rows_inserted": 13, "silver_rows_already_present": 0,
    "quarantine_candidate_rows": 2, "quarantine_duplicates_removed": 0,
    "quarantine_rows_inserted": 2,
}, QUALITY_RUN1)
check("quality metric names", sorted(audit["quality_metrics"]), sorted(QUALITY_RUN1))

# --- Silver schema: the contract columns, comment fields before lineage
check("silver columns", spark.table(TEST_SILVER).columns, [
    "event_id", "event_type", "event_timestamp", "event_date",
    "actor_id", "actor_login", "repo_id", "repo_name",
    "comment_action", "issue_id", "issue_number", "comment_target_type", "comment_id",
    "_source_file", "_source_date", "_ingested_at", "quarantine_warning",
])

# --- Silver contents
silver = {r["event_id"]: r for r in rows(TEST_SILVER)}
check("silver event_ids", sorted(silver),
      sorted(["c1", "c2", "c3", "c4", "c5", "c6", "c7", "c8", "c9", "c10",
              "c11", "c14", "c15"]))

# The three source shapes from the manual, plus the edge cases.
check("comment_target_type per row",
      {e: silver.get(e, {}).get("comment_target_type")
       for e in ["c1", "c2", "c3", "c4", "c5", "c6", "c10", "c15"]},
      {"c1": "issue", "c2": "pull_request", "c3": None, "c4": "issue",
       "c5": "issue", "c6": "pull_request", "c10": None, "c15": "issue"})

expected_warnings = {
    "c1": None,
    "c2": None,
    "c3": "MISSING: issue_id; MISSING: issue_number; UNCLASSIFIED: comment_target_type",
    "c4": "MISSING: issue_id",
    "c5": None,
    "c6": "MISSING: issue_id; MISSING: issue_number",
    "c7": "MISSING: comment_id",
    "c8": "MISSING: comment_action",
    "c9": "UNEXPECTED_VALUE: comment_action",
    "c10": "MISSING: comment_action; MISSING: issue_id; MISSING: issue_number; "
           "UNCLASSIFIED: comment_target_type; MISSING: comment_id",
    "c11": None,
    "c15": "INVALID_CAST: issue_id",
}
for event_id, expected in expected_warnings.items():
    check(f"{event_id} warning", silver.get(event_id, {}).get("quarantine_warning"), expected)

r = one_silver("c1")
check("c1 fields",
      (r.get("comment_action"), r.get("issue_id"), r.get("issue_number"), r.get("comment_id")),
      ("created", 8001, 21, 9111))

r = one_silver("c9")
check("c9 action kept unchanged", r.get("comment_action"), "edited")

check("c1 and c11 share comment_id 9111, both kept",
      sorted(e for e, r in silver.items() if r["comment_id"] == 9111), ["c1", "c11"])

r = one_silver("c14")
check("c14 winner = newer ingestion", (r.get("comment_id"), r.get("_source_file")), (1402, F2))

# --- Quarantine contents
q = rows(TEST_QUARANTINE)
check("quarantine reasons",
      sorted(r["quarantine_reason"] for r in q), ["INVALID_CAST: repo_id", "JSON_PARSE_ERROR"])
# A parse-error row is identified by its raw_json, because its parsed fields
# (including event_id) may be NULL.
parse_error = [r for r in q if r["quarantine_reason"] == "JSON_PARSE_ERROR"]
check("c12 is the parse error",
      [json.loads(r["raw_json"])["id"] for r in parse_error], ["c12"])
c13 = [r for r in q if r["event_id"] == "c13"]
check("c13 found once", len(c13), 1)
r = c13[0] if c13 else {}
check("c13 comment fields kept",
      (r.get("comment_action"), r.get("issue_id"), r.get("issue_number"),
       r.get("comment_target_type"), r.get("comment_id")),
      ("created", 8001, 21, "issue", 9001))
check("c13 original text kept",
      (r.get("issue_id_raw"), r.get("issue_number_raw"), r.get("comment_id_raw")),
      ("8001", "21", "9001"))
check("quarantine keeps raw_json", all(r["raw_json"] for r in q), True)
check("quarantine keys non-NULL", all(r["quarantine_key"] for r in q), True)

finish("B5 run 1")

## B6. Run 2: rerun the same date

In [0]:
%run ../04_issue_comment_events $test_schema="test_issue_comment" $source_date="2024-02-01" $run_id="test-run2"

## B7. Run 2 checks: idempotency, plus the final safety check

In [0]:
check_audit({
    "bronze_input_rows": 16, "silver_candidate_rows": 13,
    "silver_rows_inserted": 0, "silver_rows_already_present": 13,
    "quarantine_candidate_rows": 2, "quarantine_rows_inserted": 0,
}, QUALITY_RUN1)
check("silver rows unchanged", spark.table(TEST_SILVER).count(), 13)
check("quarantine rows unchanged", spark.table(TEST_QUARANTINE).count(), 2)

statuses = [r["status"] for r in
            spark.table(TEST_AUDIT).orderBy("run_finished_at").collect()]
check("audit history", statuses, ["SUCCEEDED", "SUCCEEDED"])

for t in PROD_TABLES:
    check(f"prod version unchanged: {t}", table_version(t), prod_versions_before[t])
finish("B7 run 2 + production safety")

## Clean up: drop the test schema

In [0]:
assert TEST_SCHEMA.startswith("test_")
spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")